## 2008 ITPA Threshold Power Database Analysis

In [10]:

import os
import numpy as np
import imas
import scipy as sp

ROOT = os.path.dirname(os.getcwd())
DIR_2008 = os.path.join(ROOT, "resources", "results", "2008")

pulse_dirs = sorted(
    os.path.join(DIR_2008, d)
    for d in os.listdir(DIR_2008)
    if d.startswith("pulse_")
)
N = len(pulse_dirs)
print(f"{N} pulses in {DIR_2008}")

7688 pulses in c:\Users\curranf\IDStools\resources\results\2008


## Load scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| PLTH | `summary/global_quantities/power_loss/value` | W |
| SPLASMA | `equilibrium/time_slice/global_quantities/surface` | m^2 |
| BT | `summary/global_quantities/b0/value` | T |
| NEL | `summary/line_average/n_e/value` | m^-3 |
| PGASA (M_eff) | `summary/volume_average/meff_hydrogenic/value` | AMU |
| IP | `summary/global_quantities/ip/value` | A |
| RGEO | `summary/global_quantities/r0/value` | m |
| TOK | `summary/machine` | — |
| SELEC2024 | `summary/tag/name` | — |

In [12]:

def first_scalar(arr):
    """Return first element of array"""
    a = np.asarray(arr, dtype=float)
    return float(a.flat[0])


PLTH    = np.full(N, np.nan)          # loss power [W]
SPLASMA = np.full(N, np.nan)          # LCFS surface area [m^2]
BT      = np.full(N, np.nan)          # vacuum B_t at R0 [T]
NEL     = np.full(N, np.nan)          # line-averaged n_e [m^-3]
MEFF    = np.full(N, np.nan)          # effective hydrogenic mass [AMU]
IP      = np.full(N, np.nan)          # plasma current [A]
RGEO    = np.full(N, np.nan)          # geometric major radius [m]
TOK     = np.empty(N, dtype=object)   # tokamak name
SEL     = np.zeros(N, dtype=bool)     # SELEC2007 flag, for inclusion in Martin 2008 analysis

for i, pulse_dir in enumerate(pulse_dirs):
    uri = f"imas:hdf5?path={pulse_dir};pulse=0"
    if i % 100 == 0:
        print(f"Processing pulse {i}: {uri}")
    with imas.DBEntry(uri, "r") as entry:
        s = entry.get("summary")
        PLTH[i]    = first_scalar(s.global_quantities.power_loss.value)
        BT[i]      = np.abs(first_scalar(s.global_quantities.b0.value))
        NEL[i]     = first_scalar(s.volume_average.n_e.value)
        MEFF[i]    = first_scalar(s.line_average.meff_hydrogenic.value)
        IP[i]      = first_scalar(s.global_quantities.ip.value)
        RGEO[i]    = first_scalar(s.global_quantities.r0.value)
        TOK[i]     = str(s.machine).strip()
        SEL[i]     = str(s.tag.name).strip() == "SELEC2007=True" # Array of bools.

        eq = entry.get("equilibrium")
        SPLASMA[i] = first_scalar(eq.time_slice[0].global_quantities.surface)
      
print("Done.")

print(f"  SELEC2007=True : {np.sum(SEL)}")

Processing pulse 0: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\2008\pulse_0000;pulse=0


IndexError: index 0 is out of bounds for size 0